# Reference workflow: deploy, benchmark, and optimize a model by hand

This notebook expands the workflow with nothing but `boto3`: every API call, parameter,
and polling loop is visible. It does not import the bundled helper scripts.

Use it to inspect instance sizing, container resolution, endpoint APIs, managed benchmarking,
inference recommendations, redeployment, and cleanup in one place.

The `SKILL.md` contracts in `.kiro/skills/` capture the same operational sequence for an
Agent-Skills-compatible coding agent.

Running example: **GPT-OSS-20B** (a strong open model that isn't one-click in JumpStart).
Nothing here is model-specific — change `MODEL_ID` / `MODEL_S3` and it all still works.


## 0. Resolve the AWS context

Nothing is hardcoded. We ask STS who we are, derive the execution role, and use SageMaker's
default bucket convention. (The `scripts/config.py` helper does exactly this — here we inline
it so you can see there's no magic.)

In [ ]:
import boto3, json, time, re, os
from datetime import datetime, timezone, timedelta

REGION = boto3.session.Session().region_name or "us-west-2"
sess   = boto3.session.Session(region_name=REGION)
sm     = sess.client("sagemaker")
sm_rt  = sess.client("sagemaker-runtime")

# Account: straight from the caller's credentials.
ACCOUNT = sess.client("sts").get_caller_identity()["Account"]

# Execution role: in SageMaker Studio the SDK knows it; otherwise set SAGEMAKER_ROLE_ARN.
ROLE = os.environ.get("SAGEMAKER_ROLE_ARN")
if not ROLE:
    try:
        import sagemaker
        ROLE = sagemaker.session.Session(boto_session=sess).get_caller_identity_arn()
    except Exception as e:
        raise RuntimeError("Set SAGEMAKER_ROLE_ARN to your SageMaker execution role ARN") from e

# Bucket: SageMaker's conventional bucket for this account/region.
BUCKET   = os.environ.get("SAGEMAKER_BUCKET", f"sagemaker-{REGION}-{ACCOUNT}")

MODEL_ID = "gpt-oss-20b"
MODEL_S3 = f"s3://{BUCKET}/models/{MODEL_ID}/"

print("region :", REGION)
print("account:", ACCOUNT)
print("role   :", ROLE)
print("bucket :", BUCKET)
print("model  :", MODEL_S3)

## 1. Pick the instance and resolve the container

Two decisions a human normally agonizes over:

1. **Instance** — pick one whose GPU memory holds the weights + KV cache. GPT-OSS-20B is
   ~13 GB (mxfp4), so a single 24 GB L4 (`ml.g6.16xlarge`) holds it comfortably. We set
   `tensor_parallel_size = GPU count` so vLLM shards across exactly the GPUs present.
2. **Container** — the newest SageMaker vLLM Deep Learning Container. We look it up **live**
   from ECR and pick the highest *vLLM version* (not the most recently pushed tag — release
   order and push order differ). The CUDA build must match the GPU generation.

In [ ]:
INSTANCE = "ml.g6.16xlarge"
NUM_GPU  = 1   # ml.g6.16xlarge has one L4; tensor_parallel_size = NUM_GPU

# Resolve the latest vLLM DLC tag from the AWS-owned DLC registry (763104351884).
ecr = sess.client("ecr")
_TAG = re.compile(r"^(\d+)\.(\d+)\.(\d+)-gpu-py312-cu(\d+)-ubuntu[\d.]+-sagemaker$")
candidates = []
for page in ecr.get_paginator("describe_images").paginate(registryId="763104351884", repositoryName="vllm"):
    for img in page["imageDetails"]:
        for tag in img.get("imageTags", []):
            m = _TAG.match(tag)
            if m:
                candidates.append((tuple(int(m.group(i)) for i in (1,2,3)), tag))
best = max(candidates)[1]
IMAGE = f"763104351884.dkr.ecr.{REGION}.amazonaws.com/vllm:{best}"
print("instance :", INSTANCE, f"(tensor_parallel_size={NUM_GPU})")
print("container:", IMAGE)

## 2. Deploy — four API calls and two polling loops

This is the part the agent collapses into *"Deploy GPT-OSS-20B."* By hand it is:

`CreateModel` → `CreateEndpointConfig` → `CreateEndpoint` (poll) →
`CreateInferenceComponent` (poll).

The vLLM server is configured **entirely** through `SM_VLLM_*` environment variables — no
Dockerfile, no custom image. An **Inference Component** declares the model's compute footprint
(memory + accelerators) and is the unit the benchmark and recommendation services target.

In [ ]:
stamp = datetime.now(timezone.utc).strftime("%y%m%d-%H%M%S")
NAME    = f"{MODEL_ID}-{stamp}"
IC_NAME = f"ic-{NAME}"

ENV = {
    "SM_VLLM_MODEL": "/opt/ml/model",                 # where the weights mount in the container
    "SM_VLLM_TENSOR_PARALLEL_SIZE": str(NUM_GPU),     # shard across all GPUs on the instance
    "SM_VLLM_MAX_NUM_SEQS": "32",                     # max sequences batched concurrently
    "SM_VLLM_MAX_MODEL_LEN": "16384",                 # context cap -> KV-cache sizing
    "SM_VLLM_ENFORCE_EAGER": "true",                  # faster cold start (drop for max throughput)
}

# (1) CreateModel — bind image + env + the S3 weights (uncompressed folder = S3Prefix/None).
sm.create_model(
    ModelName=NAME, ExecutionRoleArn=ROLE,
    PrimaryContainer={
        "Image": IMAGE, "Environment": ENV,
        "ModelDataSource": {"S3DataSource": {
            "S3Uri": MODEL_S3, "S3DataType": "S3Prefix", "CompressionType": "None"}}})
print("model created:", NAME)

# (2) CreateEndpointConfig — one production variant on the chosen instance, generous timeouts.
sm.create_endpoint_config(
    EndpointConfigName=NAME, ExecutionRoleArn=ROLE,
    ProductionVariants=[{
        "VariantName": "v1", "InstanceType": INSTANCE, "InitialInstanceCount": 1,
        "ModelDataDownloadTimeoutInSeconds": 1800,
        "ContainerStartupHealthCheckTimeoutInSeconds": 1800}])

# (3) CreateEndpoint — provision hardware, then poll until InService (~4-8 min).
sm.create_endpoint(EndpointName=NAME, EndpointConfigName=NAME)
t0 = time.time()
while True:
    st = sm.describe_endpoint(EndpointName=NAME)["EndpointStatus"]
    print(f"  endpoint: {st}  (+{int(time.time()-t0)}s)")
    if st == "InService": break
    if st == "Failed":
        # A common deployment failure: quota does not guarantee available capacity.
        raise RuntimeError(sm.describe_endpoint(EndpointName=NAME).get("FailureReason"))
    time.sleep(30)

# (4) CreateInferenceComponent — declare compute footprint, place on the variant, poll.
sm.create_inference_component(
    InferenceComponentName=IC_NAME, EndpointName=NAME, VariantName="v1",
    Specification={
        "ModelName": NAME,
        "StartupParameters": {
            "ModelDataDownloadTimeoutInSeconds": 1800,
            "ContainerStartupHealthCheckTimeoutInSeconds": 1800},
        "ComputeResourceRequirements": {
            "MinMemoryRequiredInMb": 40*1024,
            "NumberOfAcceleratorDevicesRequired": NUM_GPU}},
    RuntimeConfig={"CopyCount": 1})
while True:
    st = sm.describe_inference_component(InferenceComponentName=IC_NAME)["InferenceComponentStatus"]
    print(f"  ic: {st}  (+{int(time.time()-t0)}s)")
    if st == "InService": break
    if st == "Failed":
        raise RuntimeError(sm.describe_inference_component(InferenceComponentName=IC_NAME).get("FailureReason"))
    time.sleep(30)

print("\nENDPOINT:", NAME, "\nIC      :", IC_NAME)

## 3. Smoke test — does it actually answer?

The vLLM DLC speaks the OpenAI-style chat schema. We target the inference component by name.
A reasoning model may return its thinking in a `reasoning` field and the user-facing answer in
`content` — we print both so the behavior is visible.

In [ ]:
payload = {"messages": [{"role": "user", "content": "In one sentence, what is Amazon SageMaker AI?"}],
           "max_tokens": 256}
res = sm_rt.invoke_endpoint(EndpointName=NAME, InferenceComponentName=IC_NAME,
                            Body=json.dumps(payload), ContentType="application/json")
body = json.loads(res["Body"].read())
msg  = body["choices"][0]["message"]
print("reasoning:", (msg.get("reasoning") or "")[:200])
print("answer   :", msg.get("content"))
print("usage    :", body.get("usage"))

## 4. Baseline benchmark — the managed AIPerf job

We do **not** write a load generator. SageMaker runs **NVIDIA AIPerf** for us on managed
compute and writes the metrics to S3. Three calls: define the workload, launch the job, poll.
This result is the baseline for the later optimization comparison.

In [ ]:
WL  = f"wl-{stamp}"
JOB = f"bench-{stamp}"

# Upload the benchmark dataset. We use the curated sharegpt slice bundled with
# the benchmark skill (500 real prompts, every output budget >=32 tokens). The raw public
# feed includes a few turns with 1-3-token output budgets; a reasoning model cannot emit
# visible content in so few tokens, those requests score invalid, and AIPerf's ~1%
# validity gate fails the whole job — deterministically. The curated file keeps the
# traffic realistic while the gate measures the endpoint, not a dataset artifact.
DATASET = "../.kiro/skills/sagemaker-benchmark/datasets/sharegpt-curated.jsonl"
DATASET_S3 = f"benchmark-datasets/{stamp}/"
sess.client("s3").upload_file(DATASET, BUCKET, DATASET_S3 + "sharegpt-curated.jsonl")

workload = {
    "benchmark": {"type": "aiperf"},
    "parameters": {
        "custom_dataset_type": "single_turn",
        "input_file": "/opt/ml/input/data/datasets/sharegpt-curated.jsonl",
        "prompt_input_tokens_mean": 500, "prompt_input_tokens_stddev": 10,
        "output_tokens_mean": 256, "output_tokens_stddev": 16,
        "concurrency": 10, "request_count": 300,
        # Optional model-specific request fields, e.g. for a reasoning model:
        #   "extra_inputs": "reasoning_effort:low"
    },
}
sm.create_ai_workload_config(
    AIWorkloadConfigName=WL,
    AIWorkloadConfigs={"WorkloadSpec": {"Inline": json.dumps(workload)}},
    DatasetConfig={"InputDataConfig": [{
        "ChannelName": "datasets",   # mounted at /opt/ml/input/data/datasets/
        "DataSource": {"S3DataSource": {"S3Uri": f"s3://{BUCKET}/{DATASET_S3}"}}}]})
sm.create_ai_benchmark_job(
    AIBenchmarkJobName=JOB,
    BenchmarkTarget={"Endpoint": {"Identifier": NAME, "InferenceComponents": [{"Identifier": IC_NAME}]}},
    OutputConfig={"S3OutputLocation": f"s3://{BUCKET}/benchmark-output/"},
    AIWorkloadConfigIdentifier=WL, RoleArn=ROLE)
running = ("InProgress", "Pending", "Starting", "Stopping")
while True:
    d = sm.describe_ai_benchmark_job(AIBenchmarkJobName=JOB)
    s = d["AIBenchmarkJobStatus"]; print("  status:", s)
    if s not in running: break
    time.sleep(30)
print("done:", s, "->", d.get("OutputConfig", {}).get("S3OutputLocation") or d.get("FailureReason"))

## 4b. Read the results — download, extract, and parse the AIPerf bundle, by hand

The job wrote **one tarball** to S3. To get your numbers you download it, extract it, and
parse `profile_export_aiperf.json` yourself. The bundle also carries per-request records,
the exact prompts/answers, PNG plots, and the full AIPerf log:

```
output/
├── profile_export_aiperf.json   # aggregated metrics  <- parse this
├── profile_export_aiperf.csv    # same aggregates as CSV
├── profile_export.jsonl         # raw per-request records
├── inputs.json / outputs.json   # what was asked / what the model answered
├── benchmark_summary.txt        # completion summary
├── plots/ttft_timeline.png      # TTFT per request over the run
├── plots/ttft_over_time.png     # TTFT aggregated over the run duration
└── logs/aiperf.log              # full AIPerf execution log
```

To inspect the schema without starting a job, a sanitized completed bundle is available at
`.kiro/skills/sagemaker-benchmark/sample-output/` — open the JSON/plots there, or run
`python scripts/benchmark_results.py --local sample-output` from the benchmark skill folder.


In [ ]:
import tarfile, pathlib

# 1) Find the tarball under the job's S3 output prefix and download it.
prefix = d["OutputConfig"]["S3OutputLocation"].replace(f"s3://{BUCKET}/", "")
s3 = sess.client("s3")
keys = [o["Key"] for o in s3.list_objects_v2(Bucket=BUCKET, Prefix=prefix).get("Contents", [])]
tar_key = next(k for k in keys if k.endswith("output.tar.gz"))
work = pathlib.Path("bench-results"); work.mkdir(exist_ok=True)
s3.download_file(BUCKET, tar_key, str(work / "output.tar.gz"))

# 2) Extract it.
with tarfile.open(work / "output.tar.gz") as tar:
    tar.extractall(work)

# 3) Show what landed, then parse the aggregate metrics by hand.
for p in sorted(work.rglob("*")):
    if p.is_file() and p.name != "output.tar.gz" and "ray_tmp" not in str(p):
        print(f"  {p.relative_to(work)}")

m = json.loads((work / "profile_export_aiperf.json").read_text())
for key in ("output_token_throughput", "time_to_first_token", "inter_token_latency", "request_latency"):
    v = m[key]
    stats = "  ".join(f"{s}={v[s]:,.1f}" for s in ("avg", "p50", "p90", "p99") if s in v)
    print(f"{key:<28} [{v['unit']}]  {stats}")
# (The skill's benchmark_results.py does all of this — tree, annotations, headline table — in one command.)

## 5. Observability — CloudWatch, by hand

For an Inference Component endpoint the headline metrics live on the **InferenceComponentName**
dimension (querying EndpointName+VariantName alone returns nothing — it also needs InstanceId).
CloudWatch publishes latency in **microseconds**.

In [ ]:
cw  = sess.client("cloudwatch")
end = datetime.now(timezone.utc); start = end - timedelta(minutes=30)
dims = [{"Name": "InferenceComponentName", "Value": IC_NAME}]
def stat(metric, s):
    r = cw.get_metric_statistics(Namespace="AWS/SageMaker", MetricName=metric, Dimensions=dims,
                                 StartTime=start, EndTime=end, Period=60, Statistics=[s])
    return sorted(r["Datapoints"], key=lambda d: d["Timestamp"])
inv = stat("Invocations", "Sum")
print("invocations (total):", int(sum(p["Sum"] for p in inv)))
lat = stat("ModelLatency", "Average")
if lat: print("model latency  (avg):", round(sum(p["Average"] for p in lat)/len(lat)/1000), "ms")  # us -> ms

## 6. Optimize, redeploy, and compare

A **recommendation job** searches
serving configurations on managed compute and returns the best one — instance, copies,
container env, and the **ExpectedPerformance** — and, on the deep path, a **Model Package**
with speculative decoding (EAGLE 3) / quantization / kernel tuning applied.

**Two depths:**
- `OptimizeModel=False` — config search. Minutes to hours, depending on the workload.
- `OptimizeModel=True` — deep optimize. **Hours**, large instance (e.g. `ml.p5en.48xlarge`),
  usually capacity-reserved. Run this asynchronously and retain the completed result.

A sanitized completed result is bundled at
`.kiro/skills/sagemaker-optimize/sample-output/recommendation.json` — that is exactly what
`describe_ai_recommendation_job` returns when it finishes.

In [ ]:
REC_WL  = f"rec-wl-{stamp}"
REC_JOB = f"rec-job-{stamp}"
rec_workload = {
    "benchmark": {"type": "aiperf"},
    "parameters": {"public_dataset": "sharegpt",
                   "prompt_input_tokens_mean": 500, "prompt_input_tokens_stddev": 10,
                   "output_tokens_mean": 256, "output_tokens_stddev": 10, "concurrency": 10},
}

# A sanitized completed result is bundled for offline schema inspection
# (config search took ~70 min for GPT-OSS-20B on ml.g6.24xlarge):
#   .kiro/skills/sagemaker-optimize/sample-output/recommendation.json
sm.create_ai_workload_config(AIWorkloadConfigName=REC_WL,
    AIWorkloadConfigs={"WorkloadSpec": {"Inline": json.dumps(rec_workload)}})
sm.create_ai_recommendation_job(
    AIRecommendationJobName=REC_JOB,
    ModelSource={"S3": {"S3Uri": MODEL_S3}},
    OutputConfig={"S3OutputLocation": f"s3://{BUCKET}/recommendations/"},
    AIWorkloadConfigIdentifier=REC_WL, RoleArn=ROLE,
    PerformanceTarget={"Constraints": [{"Metric": "throughput"}]},
    ComputeSpec={"InstanceTypes": ["ml.g6.24xlarge"]},
    InferenceSpecification={"Framework": "VLLM"},
    OptimizeModel=False,
    # Deep optimize instead:  OptimizeModel=True, framework "LMI", a p5en instance, and a
    # ComputeSpec.CapacityReservationConfig with your MlReservationArns.
)
while True:
    d = sm.describe_ai_recommendation_job(AIRecommendationJobName=REC_JOB)
    s = d["AIRecommendationJobStatus"]; print("  status:", s)
    if s not in ("InProgress", "Pending", "Starting", "Stopping"): break
    time.sleep(30)

rec = d["Recommendations"][0]
print("recommended :", json.dumps(rec.get("DeploymentConfiguration"), indent=2))
print("expected    :", rec.get("ExpectedPerformance"))
print("optimized   :", [o["OptimizationType"] for o in rec.get("OptimizationDetails", [])] or "(config search only)")
print("ModelPackage:", rec.get("ModelDetails", {}).get("ModelPackageArn"))
# Deploy it: create_model(Containers=[{"ModelPackageName": <arn>}]) -> endpoint config ->
# endpoint, then benchmark again with the SAME workload for a fair before/after.

## 7. Tear down

Reverse order of creation. Run this the moment you're done.

In [ ]:
for ic in [c["InferenceComponentName"] for c in
           sm.list_inference_components(EndpointNameEquals=NAME).get("InferenceComponents", [])]:
    sm.delete_inference_component(InferenceComponentName=ic); print("deleted IC:", ic)
    while True:
        try: sm.describe_inference_component(InferenceComponentName=ic); time.sleep(10)
        except sm.exceptions.ClientError: break
for fn, label in [(lambda: sm.delete_endpoint(EndpointName=NAME), "endpoint"),
                  (lambda: sm.delete_endpoint_config(EndpointConfigName=NAME), "endpoint-config"),
                  (lambda: sm.delete_model(ModelName=NAME), "model")]:
    try: fn(); print("deleted", label)
    except sm.exceptions.ClientError as e: print("skip", label, e.response["Error"]["Code"])
print("torn down — billing stopped.")

---

### Agent-driven execution

The workflow includes context resolution, instance sizing, a container lookup, a
four-call deploy with two polling loops, a smoke test, a managed benchmark, a CloudWatch read,
a recommendation job, a redeploy, and a careful teardown.

Now ask an Agent-Skills-compatible coding agent:

> *"Deploy GPT-OSS-20B, benchmark it, find a faster config, and tear it down when we're done."*

The same sequence is captured in `.kiro/skills/*/SKILL.md` as a reusable agent contract.